In [1]:
import pandas as pd
import numpy as np
import json
import glob
import os

## Задача достать все здания вокруг станции и собрать их в один DataFrame

### Написал функции для извлечения данных из json файла

- фунцкия для извлечения amenity

In [2]:
def extract_amenity(file) -> pd.Series:
    temp = []
    
    for row in file['elements']:
        try:
            temp.append(row['tags']['amenity'])
        except KeyError as error:
            temp.append(np.nan)

    return temp

- функция для извлечения названия здания

In [3]:
def extract_name(file) -> pd.Series:
    temp = []
    
    for row in file['elements']:
        try:
            temp.append(row['tags']['name'])
        except KeyError as error:
            temp.append(np.nan)

    return temp

- функция для извлечения координат

In [4]:
def extract_coordinates(file) -> pd.Series:
    coordinat_1 = []
    coordinat_2 = []
    for row in file['elements']:
        try:
            coordinat_1.append(row['lat'])
            coordinat_2.append(row['lon'])
        except KeyError as error:
            coordinat_1.append(np.nan)
            coordinat_2.append(np.nan)
    return coordinat_1, coordinat_2

- использую библиотеку glob, которая показывает все json файлы (или любые другие файлы, которые лежат в папке)

In [11]:
json_files = glob.glob("../../overpass_data/data_about_build/*.json")
len(json_files)

911

- напичал функцию для чтения файла json

In [12]:
def read_json(file_name):
    if os.path.getsize(file_name) == 0:
        return {}

    with open(file_name, 'r', encoding='utf-8') as f:
        return json.load(f)

- создаю переменную rows для дальнейшего использования в DataFrame

In [13]:
rows = []

- написал скрипт, который проходит по каждому файлу json
- затем извлякаю нужные данные (amenity, name_build, lat, lot)
- затем записываю данные в переменную rows

In [14]:
for json_file in json_files:
    start_station_id = json_file.split("\\")[-1].split(".")[0].split('_')[0]
    file = read_json(json_file)
    if len(file) != 0:
        amenity = extract_amenity(file)
        name_build = extract_name(file)
        lat, lot = extract_coordinates(file)

        station_ids = [start_station_id] * len(amenity)
        
        for i in range(len(amenity)):
            rows.append({
                'start_station_id': station_ids[i],
                'category': amenity[i],
                'name_build': name_build[i],
                'start_lat': lat[i],
                'start_lng': lot[i]
            })

In [15]:
len(rows)

34397

- создаю DataFrame из переменной rows

In [16]:
df = pd.DataFrame(rows)
df

,start_station_id,category,name_build,start_lat,start_lng
0,30200,place_of_worship,Methodist Protestant Church,38.896501,-77.024143
1,30200,kindergarten,Georgetown Hill Early Childhood Center,38.895100,-77.023200
2,30200,kindergarten,Just Us Kids Child Development Center,38.894600,-77.020700
3,30200,theatre,Shakespeare Theatre Company - Klein Theater,38.895734,-77.022132
4,30200,cafe,Gregorys Coffee,38.895972,-77.021713
...,...,...,...,...,...
34392,33200,bench,NaN,38.894267,-77.038270
34393,33200,bench,NaN,38.897443,-77.036620
34394,33200,bench,NaN,38.897445,-77.036499
34395,33200,bench,NaN,38.896953,-77.036979


- записываю получившийся DataFrame

In [17]:
df.to_csv('../../data_additional/point_of_interest.csv', index=False)